# Reusable Template: Binary Classification with Tree-Based Models

**Project type:** Ad Click-Through Rate (CTR) Prediction / similar tabular binary problems  
**Algorithms covered:** Decision Tree (from-scratch + scikit-learn), Random Forest, Gradient Boosting (XGBoost)  
**Key techniques:** Categorical encoding, chronological split, GridSearchCV, ROC-AUC evaluation  

---

### How to use this template
1. Replace the data-loading cell with your own dataset (CSV / Parquet / database).
2. Update the feature-exclusion list and target column name.
3. Adjust the chronological split logic if your data is not time-ordered.
4. Run the notebook end-to-end; all major steps are modular and documented.
5. Extend the ensemble section with any additional models you need.

**Requirements:** `pandas`, `numpy`, `scikit-learn`, `xgboost`, `matplotlib`, `seaborn` (optional)

## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, roc_curve, classification_report
import warnings
warnings.filterwarnings('ignore')

# Optional: XGBoost
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('XGBoost not installed — gradient boosting section will use sklearn only.')

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Display options
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

## 2. Data Loading

Replace the path and `nrows` argument with your own data source.  
For the classic Avazu CTR dataset, keep `nrows=300000` for rapid iteration.

In [ ]:
# === USER: UPDATE THESE ===
DATA_PATH = 'train.csv'          # path to your labeled data
N_ROWS = 300000                  # set to None to load everything
TARGET_COL = 'click'             # binary target column
DROP_COLS = ['id', 'hour', 'device_id', 'device_ip']  # non-predictive / leakage IDs

# Load
df = pd.read_csv(DATA_PATH, nrows=N_ROWS)
print(f'Shape: {df.shape}')
print(f'Positive rate (CTR): {df[TARGET_COL].mean():.4f}')
df.head()

## 3. Feature / Target Split & Chronological Train-Test Split

Because click data is ordered by time, we **must not** shuffle randomly.  
Use the first N% of rows for training and the remainder for testing.

In [ ]:
y = df[TARGET_COL].values
X = df.drop([TARGET_COL] + DROP_COLS, axis=1).values
feature_names = df.drop([TARGET_COL] + DROP_COLS, axis=1).columns.tolist()

print(f'Number of features: {X.shape[1]}')
print(f'Feature names: {feature_names}')

# Chronological split (90 / 10)
n_train = int(len(df) * 0.9)
X_train, X_test = X[:n_train], X[n_train:]
y_train, y_test = y[:n_train], y[n_train:]

print(f'Train size: {len(y_train):,}  |  Test size: {len(y_test):,}')
print(f'Train positive rate: {y_train.mean():.4f}  |  Test positive rate: {y_test.mean():.4f}')

## 4. Categorical Encoding (One-Hot)

scikit-learn tree models require numeric input.  
Fit the encoder **only on the training set** and transform both sets.

In [ ]:
enc = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
X_train_enc = enc.fit_transform(X_train)
X_test_enc  = enc.transform(X_test)

print(f'Encoded train shape: {X_train_enc.shape}')
print(f'Encoded test shape:  {X_test_enc.shape}')
print(f'Sparsity: {1 - X_train_enc.nnz / (X_train_enc.shape[0] * X_train_enc.shape[1]):.4f}')

## 5. Helper Functions — Impurity Metrics (from scratch)

These functions implement the core split-quality measures used by CART.

In [ ]:
def gini_impurity(labels):
    """Gini impurity of a label vector."""
    if len(labels) == 0:
        return 0.0
    counts = np.unique(labels, return_counts=True)[1]
    fractions = counts / float(len(labels))
    return 1.0 - np.sum(fractions ** 2)

def entropy(labels):
    """Shannon entropy of a label vector."""
    if len(labels) == 0:
        return 0.0
    counts = np.unique(labels, return_counts=True)[1]
    fractions = counts / float(len(labels))
    return -np.sum(fractions * np.log2(fractions + 1e-12))

def weighted_impurity(groups, criterion='gini'):
    """Weighted impurity of children after a split."""
    criterion_fn = {'gini': gini_impurity, 'entropy': entropy}[criterion]
    total = sum(len(g) for g in groups)
    weighted_sum = 0.0
    for g in groups:
        weighted_sum += (len(g) / float(total)) * criterion_fn(g)
    return weighted_sum

# Quick sanity checks
print('Gini [1,1,0,1,0]:', round(gini_impurity([1,1,0,1,0]), 4))
print('Entropy [1,1,0,1,0]:', round(entropy([1,1,0,1,0]), 4))

## 6. Baseline Decision Tree (scikit-learn) + Grid Search

In [ ]:
dt = DecisionTreeClassifier(
    criterion='gini',
    min_samples_split=30,
    random_state=RANDOM_STATE
)

param_grid = {'max_depth': [3, 10, None]}

grid = GridSearchCV(
    dt,
    param_grid,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_enc, y_train)
print('Best parameters:', grid.best_params_)
print('Best CV ROC-AUC:', round(grid.best_score_, 4))

best_dt = grid.best_estimator_
pos_prob = best_dt.predict_proba(X_test_enc)[:, 1]
test_auc = roc_auc_score(y_test, pos_prob)
print(f'Test ROC-AUC (Decision Tree): {test_auc:.4f}')

## 7. Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=30,
    max_features='sqrt',
    n_jobs=-1,
    random_state=RANDOM_STATE
)

rf.fit(X_train_enc, y_train)
rf_prob = rf.predict_proba(X_test_enc)[:, 1]
rf_auc = roc_auc_score(y_test, rf_prob)
print(f'Test ROC-AUC (Random Forest): {rf_auc:.4f}')

## 8. Gradient Boosting / XGBoost

In [ ]:
if HAS_XGB:
    xgb = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='auc',
        random_state=RANDOM_STATE,
        n_jobs=-1,
        use_label_encoder=False
    )
    xgb.fit(X_train_enc, y_train)
    xgb_prob = xgb.predict_proba(X_test_enc)[:, 1]
    xgb_auc = roc_auc_score(y_test, xgb_prob)
    print(f'Test ROC-AUC (XGBoost): {xgb_auc:.4f}')
else:
    gb = GradientBoostingClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=RANDOM_STATE
    )
    gb.fit(X_train_enc, y_train)
    gb_prob = gb.predict_proba(X_test_enc)[:, 1]
    gb_auc = roc_auc_score(y_test, gb_prob)
    print(f'Test ROC-AUC (sklearn GradientBoosting): {gb_auc:.4f}')

## 9. Model Comparison & ROC Curves

In [ ]:
plt.figure(figsize=(8, 6))

# Decision Tree
fpr, tpr, _ = roc_curve(y_test, pos_prob)
plt.plot(fpr, tpr, label=f'Decision Tree (AUC={test_auc:.3f})')

# Random Forest
fpr, tpr, _ = roc_curve(y_test, rf_prob)
plt.plot(fpr, tpr, label=f'Random Forest (AUC={rf_auc:.3f})')

# Boosted model
if HAS_XGB:
    fpr, tpr, _ = roc_curve(y_test, xgb_prob)
    plt.plot(fpr, tpr, label=f'XGBoost (AUC={xgb_auc:.3f})')
else:
    fpr, tpr, _ = roc_curve(y_test, gb_prob)
    plt.plot(fpr, tpr, label=f'Gradient Boosting (AUC={gb_auc:.3f})')

# Random baseline
plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.500)')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Tree-Based Models')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Feature Importance (from best ensemble)

In [ ]:
# Use Random Forest importances (works even if XGBoost is present)
importances = rf.feature_importances_
# Note: after one-hot the feature names are expanded; for simplicity we show top indices
top_k = 15
idx = np.argsort(importances)[::-1][:top_k]

plt.figure(figsize=(10, 5))
plt.barh(range(top_k), importances[idx][::-1])
plt.yticks(range(top_k), [f'feat_{i}' for i in idx[::-1]])
plt.xlabel('Importance')
plt.title(f'Top {top_k} Feature Importances (Random Forest)')
plt.tight_layout()
plt.show()

# Optional: map back to original categorical columns if needed
# (requires inspecting enc.categories_ and the original feature order)

## 11. Package the Pipeline for Reuse

In [ ]:
import joblib

# Save encoder + best model together
artifact = {
    'encoder': enc,
    'model': rf,               # or xgb / best_dt
    'feature_names': feature_names,
    'drop_cols': DROP_COLS,
    'target_col': TARGET_COL
}

joblib.dump(artifact, 'ctr_tree_pipeline.joblib')
print('Pipeline saved to ctr_tree_pipeline.joblib')

# Example prediction function
def predict_ctr(raw_df, artifact_path='ctr_tree_pipeline.joblib'):
    """
    raw_df : DataFrame with the same columns as the original training features
             (before dropping DROP_COLS)
    Returns predicted click probability.
    """
    art = joblib.load(artifact_path)
    X = raw_df.drop(art['drop_cols'], axis=1, errors='ignore').values
    X_enc = art['encoder'].transform(X)
    return art['model'].predict_proba(X_enc)[:, 1]

## 12. Next Steps / Extensions

- Add target encoding or embedding for ultra-high-cardinality features.
- Experiment with `class_weight='balanced'` or SMOTE for severe imbalance.
- Calibrate probabilities (`CalibratedClassifierCV`) if downstream decisions need well-calibrated scores.
- Monitor feature drift and retrain on a rolling window.
- Deploy the joblib artifact behind a simple FastAPI / Flask endpoint.

---
*Template derived from Chapter 3 of “Python Machine Learning by Example” (4th ed.) — adapted for reuse across similar binary tabular problems.*